# Single-Qutrit Simulation of Hilbert-Space Inner-Product Change

This notebook implements quantum process tomography (QPT) for the single-qutrit simulation
of a non-unitary inner-product (IP) change on a Rigetti superconducting processor.

The non-unitary action is embedded into the native qutrit subspace of a transmon qubit
(qubit 45 on Aspen-11), exploiting the |0>, |1>, |2> levels. Process tomography is
performed using a custom readout procedure that classifies all three qutrit states and
applies readout error mitigation (REM) via confusion matrix inversion.

Raw counts and REM-corrected counts for each of the 40 metric operators are saved to
`data/raw_tomography/qutrit_raw_counts/`.

The 40 metric operators are parameterised by $(r, \\theta, \\phi)$ as defined in the
manuscript. Each corresponds to a 2x2 positive-definite matrix with largest eigenvalue 1.

In [ ]:
import os
import csv
from math import pi

import numpy as np
from sklearn import svm
from sklearn.metrics import confusion_matrix

from pyquil import get_qc, Program
from pyquil.api import QuantumComputer
from pyquil.gates import Gate, Qubit, RX, RZ, MEASURE

from forest.benchmarking.compilation import basic_compile
from forest.benchmarking.observable_estimation import (
    ObservablesExperiment,
    generate_experiment_programs,
)
from forest.benchmarking.tomography import generate_process_tomography_experiment

## Hardware initialisation

In [ ]:
lattice = "Aspen-11"
qc = get_qc(lattice)
cals = qc.compiler.get_calibration_program()

# Qubit index used as qutrit (levels |0>, |1>, |2>)
qubit = 45
pyquil_ver = 3
num_shots = 10000

## Metric operator parameter grid

Generates a grid of 40 points in $(r, \\theta, \\phi, \\alpha)$ space. Each point
parameterises a 2x2 positive-definite metric operator $\\eta$ with largest eigenvalue 1,
as defined in the manuscript (Eq. X). The parameter $\\alpha$ is derived from $r$ and
controls the mixing angle of the qutrit embedding unitary.

In [ ]:
def eta_grid():
    """Generate a grid of 40 metric operator parameter vectors [r, theta, phi, alpha].

    The first 20 points sample two delta values at five nu values with phi = -pi/2.
    The next 20 points are related to the first 20 by a Hadamard transform, with
    phi = pi/2 and theta = pi/2.

    Returns:
        list of [r, theta, phi, alpha] parameter vectors (length 40).
    """
    grid = []
    nu_array = np.linspace(0.05, 0.95, 5)

    # First 20 points: phi = -pi/2 and pi/2, theta varies with delta
    for delta_array, phi in [
        (np.linspace(-pi/4, pi/4, 2), -pi/2),
        (np.linspace(3*pi/4, 5*pi/4, 2), pi/2),
    ]:
        for nu in nu_array:
            for delta in delta_array:
                r = (1 - nu) / (1 + nu)
                theta = pi/2 - delta if phi == -pi/2 else -pi/2 + delta
                alpha = 2 * np.arccos(np.sqrt(1 - np.abs(r)))
                grid.append([r, theta, phi, alpha])

    # Last 20 points: theta = pi/2, phi varies with delta (Hadamard-related metrics)
    delta_array = np.linspace(pi/4, 7*pi/4, 4)
    for nu in nu_array:
        for delta in delta_array:
            r = (1 - nu) / (1 + nu)
            theta = pi / 2
            phi = 3*pi/2 - delta
            alpha = 2 * np.arccos(np.sqrt(1 - np.abs(r)))
            grid.append([r, theta, phi, alpha])

    return grid

## Qutrit readout calibration

The transmon's third level |2> is accessed via a native `RX_12(pi)` pulse on the 1-2
transition. The readout pulse parameters below were optimised for qubit 45 on Aspen-11
to maximise SNR across all three qutrit states. The readout frequency is shifted by
`f_ro` relative to the default to improve discrimination of |2>.

A linear SVM classifier is trained on IQ data from all three prepared states and its
confusion matrix is used to perform readout error mitigation (REM).

In [ ]:
def opt_ro_instruction1(calibrations, qubit, shift, dur, scale):
    """Build a custom readout calibration program with tunable frequency, duration, and amplitude.

    Modifies the default Rigetti MEASURE calibration to return raw IQ values
    (REAL[2]) instead of classified bits, and adjusts the readout pulse frequency,
    duration, and amplitude to improve qutrit state discrimination.

    Args:
        calibrations: QPU calibration Program from qc.compiler.get_calibration_program().
        qubit: qubit index.
        shift: frequency shift relative to default readout frequency (Hz).
        dur: readout pulse duration (s), must be aligned to 4-sample boundary.
        scale: readout pulse amplitude scaling factor.

    Returns:
        A pyQuil Program implementing the modified readout calibration.
    """
    from copy import deepcopy
    from pyquil.quilbase import Pulse, Capture
    from pyquil.gates import SHIFT_FREQUENCY

    measure_cal = deepcopy(calibrations.get_calibration(MEASURE(qubit, "")))
    new_meas = deepcopy(measure_cal)

    # Remove 1->0 pre/post rotations to expose raw IQ output
    new_meas.instrs.pop(2)
    new_meas.instrs.pop(-2)

    pulse = next(i for i in new_meas.instrs if isinstance(i, Pulse))
    capture = next(i for i in new_meas.instrs if isinstance(i, Capture))

    # Apply custom duration and amplitude
    pulse.waveform.duration = dur
    capture.kernel.duration = dur
    pulse.waveform.scale = scale

    target_name = measure_cal.memory_reference.name
    new_meas.instrs = new_meas.instrs[:-4] + [new_meas.instrs[-1]]
    new_meas.instrs[-2].freeform_string = target_name

    new_ro = deepcopy(new_meas)
    new_ro.instrs = (new_meas.instrs[:2]
                     + [SHIFT_FREQUENCY(pulse.frame, shift)]
                     + [new_meas.instrs[2]])
    new_ro.instrs += [SHIFT_FREQUENCY(capture.frame, shift)] + new_meas.instrs[3:]
    return new_ro

In [ ]:
def readout_data(quantumcomputer, ro_instruction, excite2_inst, qubit, pyquil_ver):
    """Collect IQ readout data for qutrit states |0>, |1>, and |2>.

    Prepares each of the three qutrit basis states in turn, runs the custom
    readout calibration program, and accumulates raw IQ data and state labels
    for classifier training.

    Args:
        quantumcomputer: quantum computer object on which to run programs.
        ro_instruction: custom readout calibration Program (from opt_ro_instruction1).
        excite2_inst: Program that prepares the |2> state from |0>.
        qubit: qubit index.
        pyquil_ver: pyQuil major version (2 or 3); controls readout_data access.

    Returns:
        X: (6000, 2) array of IQ data points (2000 shots per state).
        y: (6000,) array of integer state labels {0, 1, 2}.
    """
    measure_inst = ro_instruction

    def run_state_prep(prep_prog):
        prog = prep_prog.copy()
        ro = prog.declare(name="ro", memory_type="REAL", memory_size=2)
        prog += MEASURE(qubit, ro)
        prog += measure_inst
        prog.num_shots = 2000
        exe = quantumcomputer.compiler.native_quil_to_executable(prog)
        if pyquil_ver == 3:
            return quantumcomputer.run(exe).readout_data["ro"]
        elif pyquil_ver == 2:
            return quantumcomputer.run(exe)

    shots_0 = run_state_prep(Program())

    p1 = Program()
    p1 += RX(np.pi, qubit)
    shots_1 = run_state_prep(p1)

    shots_2 = run_state_prep(excite2_inst)

    X = np.vstack((shots_0, shots_1, shots_2))
    y = np.hstack((
        np.zeros(len(shots_0)),
        np.ones(len(shots_1)),
        2. * np.ones(len(shots_2)),
    ))
    return X, y

In [ ]:
def readout_analysis(IQ_data, state_label,
                     fidelity_output=False, plot=False,
                     save=False, classifier=False):
    """Fit a linear SVM classifier to qutrit IQ data and compute the confusion matrix.

    Trains a linear support vector machine on the IQ calibration data to distinguish
    qutrit states {0, 1, 2}. The resulting confusion matrix is used downstream for
    readout error mitigation (REM).

    Args:
        IQ_data: (N, 2) array of raw IQ data points.
        state_label: (N,) array of integer state labels {0, 1, 2}.
        fidelity_output: if True, return only the classification fidelity.
        plot: unused; retained for interface compatibility.
        save: unused; retained for interface compatibility.
        classifier: if True, return (fidelity, dict) where dict contains
            the fitted classifier and confusion matrix.

    Returns:
        If classifier=True: (fidelity, {"classifier": clf, "confusion_matrix": conf_matrix})
        If fidelity_output=True: fidelity (float)
        Otherwise: fidelity (float)
    """
    X = IQ_data
    y = state_label

    clf = svm.SVC(kernel="linear")
    clf.fit(X, y)
    conf_matrix = confusion_matrix(y, clf.predict(X)) / 2000
    fidelity = clf.score(X, y)

    if fidelity_output:
        return fidelity
    elif classifier:
        return fidelity, {"classifier": clf, "confusion_matrix": conf_matrix}
    else:
        return fidelity

## Readout calibration setup

The readout pulse parameters below were optimised for qubit 45 on Aspen-11 to maximise
SNR across all three qutrit states. The readout frequency is shifted by `f_ro` relative
to the Rigetti default to improve discrimination of |2>.

The program `excite2_prog` prepares |2> from |0> via sequential pi pulses on the 0-1
and 1-2 transitions, and is used during classifier training.

In [ ]:
# Optimised readout pulse parameters for qutrit discrimination on qubit 45
a_ro = 0.2511886431509585   # readout pulse amplitude (a.u.)
f_ro = -1.2e6               # readout frequency shift relative to default (Hz)
t_ro = 1.62e-6              # readout pulse duration (s)
samplingrate_ro = 1e9       # ADC sampling rate (Hz)

# Align readout duration to 4-sample boundary required by the hardware
t_ro = 4 * np.round(t_ro * samplingrate_ro / 4) / samplingrate_ro

# Build the custom readout calibration program
final_measure_cal = opt_ro_instruction1(cals, qubit, f_ro, t_ro, a_ro)

# Program to excite |0> -> |1> -> |2> for classifier training
excite2_prog = Program()
excite2_prog += RX(np.pi, qubit)
excite2_prog += Program(f"RX_12(pi) {qubit}")

## Native qutrit gate definitions

The 1-2 transition gates (`RX_12`, `RZ_12`) are not part of the standard pyQuil gate
set and are accessed as custom named gates. Phase correction angles (`phase0_cor`,
`phase2_cor`) account for frame offsets between the 0-1 and 1-2 transition drives,
calibrated for qubit 45.

In [ ]:
# Native qutrit gate wrappers for the 1-2 transition
X180_12 = Gate("RX_12", params=[np.pi], qubits=[Qubit(index=qubit)])
X90_12 = Gate("RX_12", params=[np.pi / 2], qubits=[Qubit(index=qubit)])

def Z_12(angle):
    """Virtual Z gate on the 1-2 transition (implemented as a frame shift)."""
    return Gate("RZ_12", params=[angle], qubits=[Qubit(index=qubit)])

# Phase correction for X90 on 1-2 transition (accounts for 0-1/1-2 frame offset on qubit 45)
phase0_cor = -1.6  # degrees
def X90_12_cor():
    """Phase-corrected X90 gate on the 1-2 transition."""
    prog = Program()
    prog += RZ(phase0_cor * np.pi / 180., qubit)
    prog += X90_12
    return prog

# Phase correction for X90 on 0-1 transition when preceded by 1-2 operations
phase2_cor = 3.7  # degrees
def X90_01_cor():
    """Phase-corrected X90 gate on the 0-1 transition."""
    prog = Program()
    prog += Z_12(phase2_cor * np.pi / 180.)
    prog += RX(np.pi / 2, qubit)
    return prog

## QPT measurement functions

In [ ]:
def run_qpt_readout(qc, program, meas_qubit, num_shots=10000):
    """Run a single QPT tomography program and return raw and REM-corrected qutrit counts.

    Collects IQ readout data, classifies each shot into qutrit states {0, 1, 2} using
    a linear SVM trained on calibration data, then applies readout error mitigation
    (REM) by inverting the confusion matrix.

    Args:
        qc: quantum computer object on which to run the program.
        program: pyQuil Program implementing one QPT tomography setting.
        meas_qubit: index of the qubit to measure.
        num_shots: number of shots to collect.

    Returns:
        A length-6 array [raw_0, raw_1, raw_2, rem_0, rem_1, rem_2], where
        raw_i are the raw counts (normalised by num_shots) and rem_i are the
        REM-corrected counts for qutrit state |i>.
    """
    # Train classifier on current hardware state
    qutrit_IQ, qutrit_label = readout_data(
        qc, final_measure_cal, excite2_prog, qubit, pyquil_ver
    )
    _, classifiers = readout_analysis(
        qutrit_IQ, qutrit_label, classifier=True
    )

    # Append readout instructions to program
    prog = program.copy()
    ro = prog.declare(name="ro", memory_type="REAL", memory_size=2)
    prog += MEASURE(meas_qubit, ro)
    prog += final_measure_cal
    prog.num_shots = num_shots

    executable = qc.compiler.native_quil_to_executable(prog)
    result_unclassified = qc.run(executable).readout_data['ro']

    # Classify IQ data into qutrit states {0, 1, 2}
    result_classified = np.array(classifiers["classifier"].predict(result_unclassified))

    # Compute normalised raw counts
    _, counts = np.unique(result_classified, return_counts=True)
    counts = [float(c) / num_shots for c in counts]

    # Apply REM via confusion matrix inversion
    err_mig = np.linalg.inv(classifiers['confusion_matrix']).T @ counts

    return np.hstack((np.array(counts), err_mig))

In [ ]:
def estimate_qpt_observables(qc, obs_expt, num_shots=10000, use_basic_compile=True):
    """Run all QPT tomography programs and collect qutrit count data.

    This is a modified version of estimate_observables from
    forest.benchmarking.observable_estimation, adapted to collect raw IQ
    count data for qutrit state discrimination rather than Pauli expectation values.

    Args:
        qc: quantum computer object on which to run experiments.
        obs_expt: process tomography experiment specification.
        num_shots: number of shots per tomography setting.
        use_basic_compile: whether to use basic_compile instead of the default compiler.

    Returns:
        list of length-6 arrays [raw_0, raw_1, raw_2, rem_0, rem_1, rem_2],
        one per tomography setting.
    """
    if use_basic_compile:
        old_method = qc.compiler.quil_to_native_quil
        qc.compiler.quil_to_native_quil = basic_compile

    programs, meas_qubits = generate_experiment_programs(
        obs_expt, active_reset=False, use_basic_compile=False
    )
    results = [
        run_qpt_readout(qc, prog, meas_qs[0], num_shots)
        for prog, meas_qs, _ in zip(programs, meas_qubits, obs_expt)
    ]

    if use_basic_compile:
        qc.compiler.quil_to_native_quil = old_method

    return results

## Run process tomography for all 40 metric operators

In [ ]:
# Output directory (relative to repository root)
OUTPUT_DIR = "data/raw_tomography/qutrit_raw_counts"
os.makedirs(OUTPUT_DIR, exist_ok=True)

grid = eta_grid()

for idx, vect in enumerate(grid):
    theta = vect[1]
    phi   = vect[2]
    alpha = vect[3]
    gamma = -np.pi + alpha

    # Build the qutrit circuit for this metric operator.
    # The decomposition uses X90 gates on the 0-1 and 1-2 transitions interleaved
    # with virtual Z rotations to implement the target SU(3) unitary.
    process = Program()
    process += Program(f"""DEFCAL RZ_12(%theta) {qubit}:
    SHIFT-PHASE {qubit} "rf_f12" -1.0*%theta
    """)
    process += RZ(-phi, qubit);          process += Z_12(phi / 2)
    process += RZ(-np.pi, qubit);        process += Z_12(np.pi / 2)
    process += X90_01_cor()
    process += RZ(np.pi + theta, qubit); process += Z_12(-(np.pi + theta) / 2)
    process += X90_01_cor()
    process += RZ(np.pi / 2, qubit);     process += Z_12(-np.pi)
    process += X90_12_cor()
    process += RZ(-(np.pi - gamma) / 2, qubit); process += Z_12(np.pi - gamma)
    process += X90_12_cor()
    process += RZ(-np.pi, qubit);        process += Z_12(np.pi / 2)
    process += X90_01_cor()
    process += RZ(np.pi - theta, qubit); process += Z_12(-(np.pi - theta) / 2)
    process += X90_01_cor()
    process += RZ(phi, qubit);           process += Z_12(-phi / 2)

    # Run process tomography
    tomography_experiment = generate_process_tomography_experiment(process, [qubit])
    results = estimate_qpt_observables(
        qc, tomography_experiment, num_shots, use_basic_compile=False
    )

    # Save raw and REM-corrected qutrit counts
    out_path = os.path.join(OUTPUT_DIR, f"qutrit_tomography_metric_{idx+1:02d}.csv")
    with open(out_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["raw_0", "raw_1", "raw_2",
                         "mitigated_0", "mitigated_1", "mitigated_2"])
        writer.writerows(results)

print("Done. Qutrit tomography data saved to", OUTPUT_DIR)